# slides_v2 — WorkInProgress v9 data and figures

Run all cells on **hhw9l84**, from **~/Developer/rfmapping**, with **~/.virtualenvs/rfmapping/bin/python**. The notebook reads saved recording sources and detections and generates the derived data, SVGs and opaque-white PNGs in **slides_v2_output/**. It does not rerun sorting, RF detection or raw-data preparation, or modify recording files.

| Original v9 slides | Generated content |
| --- | --- |
| 11–13 | Units 269, 305, 295: HD tuning and ON/OFF RF maps in rectangular and annular views |
| 15–18 | Unmasked response maps with Hz scales, saved masks and cumulative counts |
| 19 | M22 RF coverage |
| 20 | Seven-mouse RF coverage and HD/RF fractions from the same seven recordings |
| 22–24 | Grayscale HD/RF sorting and all-height 2D-to-1D RF projection |
| 25–26 | Code-drawn schematics, HD positions and 25-region histogram |
| Sample RF | Unit 269 annular ON map with a forward-facing mouse in the center |
| 26, new 27 | One peak-bin direction per cell and the requested seven-mouse polar panel |

All RF tuning curves sum **every elevation row** in 0–200 ms. The original RF-only zero-bin filter defines membership, so the M22 cohort remains 18 cells. Each new peak polar plot uses the maximum HD bin plus the maximum all-height RF bin, modulo 360°, with the TC comparison notebook's 15-bin histogram. Saved region centers and the existing 25-region plot remain distinct.

Slides 1–10 are introductory artwork and native PowerPoint schematics. This notebook generates the experimental data figures used from slide 11 onward.

In [ ]:
from pathlib import Path
from hashlib import file_digest
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

from HD.profiles import load_hd_tc, hd_profiles
from RF.paths import rf_source_path
from RF.profiles import prep_rf_tc, rf_centers
from RF.rflocate import load_rf, load_rfmap, rf_result_path
from Utils.plotting import LIGHT_PLOT_STYLE
from Utils.statistic_utils import rayleigh_test, smooth_tuning_curve

%matplotlib inline

root = Path('/mnt/senzailab/Kai/#Recording')
mouse, date, probe = 'm22', '261008', 'A'
hd_session, rf_session = 9, 2
base = root / mouse / date
output_dir = Path.cwd() / 'slides_v2_output'
output_dir.mkdir(exist_ok=True)
example_units = [256, 269, 305, 295]
sigma_deg = 6.
histogram_bins = 15
arrow_hd_bins = 180
arrow_hd_smoothing_deg = 12.

plt.rcParams.update({
    **LIGHT_PLOT_STYLE, 'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'savefig.facecolor': 'white', 'savefig.transparent': False,
    'text.color': '#202020', 'axes.labelcolor': '#202020',
    'xtick.color': '#202020', 'ytick.color': '#202020',
    'svg.fonttype': 'none', 'font.size': 11,
})


def save_figure(fig, stem):
    for extension in ('svg', 'png'):
        fig.savefig(output_dir / f'{stem}.{extension}', dpi=180,
                    facecolor='white', transparent=False, bbox_inches='tight')


def file_record(path):
    with path.open('rb') as stream:
        digest = file_digest(stream, 'sha256').hexdigest()
    return dict(path=str(path), sha256=digest)

from HD.profiles import load_hd_comparison, prep_hd_tc
from Utils.direction_comparison import peak_angles, concat
from scripts.slides_v9_figures import (
    example_figure, rectangle_panel_figure, sample_rf_polar_figure,
    cumulative_mask_figure, prepare_count_coverage,
    pool_count_coverage, count_coverage_figure, sorting_figure,
    peak_sum_data, peak_sum_figure, seven_mice_figure,
)
from scripts.export_m22_rf_projection import projection_data, projection_export, projection_figure

# One paired recording per requested mouse. M15 includes both jointly recorded probes.
comparison_recordings = [
    ('m14', '260609', 1, 3, ('A',)),
    ('m15', '260630', 1, 3, ('A', 'B')),
    ('m19', '260827', 9, 2, ('A',)),
    ('m21', '261006', 9, 3, ('A',)),
    ('m22', '261008', 9, 2, ('A',)),
    ('m23', '261015', 9, 2, ('A',)),
    ('m24', '261104', 10, 2, ('A',)),
]
from scripts.export_m22_rf_sample_variants import rectangle_figure


## Load saved M22 inputs
HD session 9 and RF session 2, Probe A. Saved excitatory detections define membership.

In [ ]:
hd_path = base / f'{date}_{hd_session}/data/tuning_curves/Probe{probe}/tuning_curves.tc'
rf_paths = {
    polarity: rf_source_path(base / f'{date}_{rf_session}', probe, root_kind=kind)
    for polarity, kind in (('ON', 'rfmapping'), ('OFF', 'rfmapping_off'))
}
detector_paths = {polarity: rf_result_path(path, rf_type='excitatory')
                  for polarity, path in rf_paths.items()}
hd_saved = load_hd_tc(hd_path)
detected = {polarity: load_rf(path) for polarity, path in detector_paths.items()}
for result in detected.values():
    assert result.manifest['parameters']['alternative'] == 'greater'
    np.testing.assert_allclose(result.manifest['time_range_s'], [0, .2])

## M22 cohort and saved RF region centers
Retain the 18-cell cohort. The region plots use HD profiles at 180 bins with circular Gaussian σ=12° on counts and occupancy before rate division. The new peak-bin polar plot uses the original 30-bin HD comparison.

In [ ]:
hd = hd_profiles(hd_saved, probe=probe, bins=30, smoothing_deg=0, hd_class=3)
hd.index = hd.index.get_level_values('unit_id')
rf_units = pd.Index(detected['ON'].unit_ids[detected['ON'].center_2d.any(axis=(1, 2))])
hd_peaks = hd.loc[hd.index.intersection(rf_units)].idxmax(axis=1).sort_values(kind='stable')
unit_ids = hd_peaks.index
on = detected['ON']
x = np.asarray(on.manifest['x_positions'], dtype=float)
y = np.asarray(on.manifest['y_positions'], dtype=float)
centers = rf_centers(on, x_positions=x, y_positions=y, probe=probe)
region_pairs = centers.loc[pd.MultiIndex.from_product([[probe], unit_ids])].reset_index()
region_pairs['rf_region'] = region_pairs.groupby('unit_id', sort=False).cumcount() + 1
region_pairs['hd_peak_deg'] = region_pairs.unit_id.map(hd_peaks)
region_pairs['direction_deg'] = (region_pairs.hd_peak_deg + region_pairs.rf_x_deg) % 360
region_pairs['rf_type'] = 'excitatory'
reference_pairs = region_pairs.copy()
reference_pairs.to_csv(output_dir / 'm22_261008_hd_rf_region_pairs_reference_30bins.csv', index=False)
arrow_hd = hd_profiles(hd_saved, probe=probe, bins=arrow_hd_bins,
                       smoothing_deg=arrow_hd_smoothing_deg, hd_class=3)
arrow_hd.index = arrow_hd.index.get_level_values('unit_id')
arrow_hd = arrow_hd.loc[unit_ids]
arrow_hd_peaks = arrow_hd.idxmax(axis=1)
region_pairs['hd_peak_deg'] = region_pairs.unit_id.map(arrow_hd_peaks)
region_pairs['direction_deg'] = (region_pairs.hd_peak_deg + region_pairs.rf_x_deg) % 360
arrow_hd.to_csv(output_dir / 'm22_261008_hd_180bins_smooth12deg_hz.csv')
n_cells, n_regions = region_pairs.unit_id.nunique(), len(region_pairs)
assert (n_cells, n_regions) == (18, 25)
region_pairs.to_csv(output_dir / 'm22_261008_hd_rf_region_pairs.csv', index=False)
display(region_pairs)

## Generate response maps and the all-height RF tuning data
Use raw counts for the projection and true presentation-normalized Hz for ON/OFF maps. Each azimuth includes all seven elevation rows, including bins outside the saved RF mask.

In [ ]:
example_curves = {unit: smooth_tuning_curve(hd.loc[unit], sigma=1) for unit in example_units}
example_responses = {unit: {} for unit in example_units}
example_counts = {unit: {} for unit in example_units}
example_masks = {unit: {} for unit in example_units}
for polarity, path in rf_paths.items():
    maps = load_rfmap(path)
    presentations = maps.by_unit_id(example_units[0]).to_firing_rate(
        reconstruct_presentations=True).presentation_counts
    for unit in example_units:
        native = maps.by_unit_id(unit)
        example_counts[unit][polarity] = native.sum(0, .2)
        example_responses[unit][polarity] = native.to_firing_rate(
            presentation_counts=presentations).mean_rate(0, .2)
        example_masks[unit][polarity] = detected[polarity].for_map(example_responses[unit][polarity])
    if polarity == 'ON':
        rf_profiles_saved = pd.DataFrame(
            [maps.by_unit_id(int(unit)).sum(0, .2).sum_to_1d(axis='x', rf_only=False).to_1d_array()
             for unit in unit_ids],
            index=[f'{mouse}:{date}:{probe}:{unit}' for unit in unit_ids],
            columns=maps[0].x_positions)
        rf_profiles_saved.attrs.update(label='M22 all-height RF', range=[-180, 180],
                                      response_units='spike_count', response_kind='RF')
        measured = np.asarray(maps[0].metadata['occupancyTimeSec']).reshape(len(y), len(x)) > 0
        m22_coverage = prepare_count_coverage(on, measured, unit_ids=hd.index)
    del maps
rf_profiles_saved.to_csv(output_dir / 'm22_261008_rf_all_height_native_counts.csv', index_label='unit_id')
np.savez_compressed(output_dir / 'm22_261008_sample_saved_masks.npz',
    unit_ids=example_units, x=x, y=y,
    on_masks=np.stack([example_masks[unit]['ON'] for unit in example_units]),
    off_masks=np.stack([example_masks[unit]['OFF'] for unit in example_units]),
    on_hz=np.stack([example_responses[unit]['ON'].to_2d_array() for unit in example_units]),
    off_hz=np.stack([example_responses[unit]['OFF'].to_2d_array() for unit in example_units]),
    on_counts=np.stack([example_counts[unit]['ON'].to_2d_array() for unit in example_units]),
    hd_hz=np.stack([example_curves[unit].to_numpy() for unit in example_units]))

## Slide 11 — unit 269: HD, rectangular RF and annular RF
Both RF views use the same unsmoothed native cells and saved ON mask.

In [ ]:
fig = example_figure(269, example_curves[269], example_responses[269], example_masks[269]['ON'])
save_figure(fig, 'v9_slide11_unit269_hd_rf')
display(fig)
plt.close(fig)
for polarity in ('ON', 'OFF'):
    fig = rectangle_panel_figure(example_responses[269][polarity])
    save_figure(fig, f'v9_unit269_{polarity.lower()}_rectangular_panel')
    plt.close(fig)


## Slide 12 — unit 305: HD, rectangular RF and annular RF
Both RF views use the same unsmoothed native cells and saved ON mask.

In [ ]:
fig = example_figure(305, example_curves[305], example_responses[305], example_masks[305]['ON'])
save_figure(fig, 'v9_slide12_unit305_hd_rf')
display(fig)
plt.close(fig)
for polarity in ('ON', 'OFF'):
    fig = rectangle_panel_figure(example_responses[305][polarity])
    save_figure(fig, f'v9_unit305_{polarity.lower()}_rectangular_panel')
    plt.close(fig)


## Slide 13 — unit 295: HD, rectangular RF and annular RF
Both RF views use the same unsmoothed native cells and saved ON mask.

In [ ]:
fig = example_figure(295, example_curves[295], example_responses[295], example_masks[295]['ON'])
save_figure(fig, 'v9_slide13_unit295_hd_rf')
display(fig)
plt.close(fig)
for polarity in ('ON', 'OFF'):
    fig = rectangle_panel_figure(example_responses[295][polarity])
    save_figure(fig, f'v9_unit295_{polarity.lower()}_rectangular_panel')
    plt.close(fig)


## Slides 15–18 — unmasked responses, saved masks and cumulative RF counts
Each unmasked ON response map has its native firing-rate scale in Hz. Binary masks retain the saved membership, and the combined map counts masks in sample order 256 → 269 → 305 → 295.


In [ ]:
sample_counts = np.cumsum(np.stack([example_masks[unit]['ON'] for unit in example_units]), axis=0)
np.savez_compressed(output_dir / 'm22_261008_sample_cumulative_counts.npz',
                    unit_ids=example_units, x=x, y=y, counts=sample_counts)
for slide, unit, values in zip(range(15, 19), example_units, sample_counts):
    fig = rectangle_figure(example_responses[unit]['ON'], example_masks[unit]['ON'],
                           unit, outline=False)
    save_figure(fig, f'v9_slide{slide}_unit{unit}_on_response')
    display(fig)
    plt.close(fig)
    fig = cumulative_mask_figure(example_responses[unit]['ON'], example_masks[unit]['ON'], values, unit)
    save_figure(fig, f'v9_slide{slide}_unit{unit}_masks')
    display(fig)
    plt.close(fig)
scale_data = {'units': {str(unit): dict(
    on_min=float(example_responses[unit]['ON'].to_2d_array().min()),
    on_max=float(example_responses[unit]['ON'].to_2d_array().max()),
    count_max=int(sample_counts[i].max())) for i, unit in enumerate(example_units)}}
(output_dir / 'v9_native_scale_data.json').write_text(json.dumps(scale_data, indent=2) + '\n')


## Slide 17 — M22 RF coverage
The same 18 HD-class-3 units with saved excitatory ON RFs used in the M22 comparison panels. Exclude non-HD RF-positive unit 1136. Native Gaussian σ=6° smoothing is a display operation.

In [ ]:
from matplotlib.ticker import MaxNLocator

assert m22_coverage['n_rf'] == len(unit_ids) == 18
assert m22_coverage['n_assessed'] == np.isin(on.unit_ids, hd.index).sum()
np.savez_compressed(output_dir / 'm22_261008_population_data.npz', **m22_coverage)
fig = count_coverage_figure(m22_coverage)
fig.axes[1].yaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
fig.axes[2].xaxis.set_major_locator(MaxNLocator(nbins=4, integer=True))
save_figure(fig, 'm22_261008_population_coverage')
display(fig)
plt.close(fig)

## Original slide 20 (current slide 18) — seven-mouse coverage map
Use M14, M15 A+B, M19, M21, M22, M23 and M24, with saved HD class 3 selection and no additional anatomy restriction, matching poster.ipynb. Count the 129 HD units with saved excitatory ON RF masks. Smooth each native grid at σ=6° before summing on the exact union of cell edges. The top marginal counts distinct units with an RF at any elevation in each azimuth column; the right marginal counts distinct units with an RF at any azimuth in each elevation row. Both use unsmoothed saved binary masks, counting each unit once per column or row. The axes label the true boundaries: azimuth −180…180° and elevation +45…−45°.

In [ ]:
coverage_panels, coverage_sources = [], []
for mouse_id, rec_date, hd_id, rf_id, probes in comparison_recordings:
    for rec_probe in probes:
        source = rf_source_path(root / mouse_id / rec_date / f'{rec_date}_{rf_id}', rec_probe)
        result_path = rf_result_path(source, rf_type='excitatory')
        result, maps = load_rf(result_path), load_rfmap(source)
        shape = result.mask_2d.shape[1:]
        measured = np.asarray(maps[0].metadata['occupancyTimeSec']).reshape(shape) > 0
        hd_file = root / mouse_id / rec_date / f'{rec_date}_{hd_id}/data/tuning_curves/Probe{rec_probe}/tuning_curves.tc'
        saved_hd = load_hd_tc(hd_file)
        hd_ids = np.asarray(saved_hd['unit_id'])[np.asarray(saved_hd['unit_data']['hd_class']) == 3]
        panel = prepare_count_coverage(result, measured, unit_ids=hd_ids)
        coverage_panels.append(panel)
        key = f'{mouse_id}_{rec_date}_Probe{rec_probe}'
        np.savez_compressed(output_dir / f'v9_coverage_{key}.npz', **panel)
        coverage_sources.append(dict(mouse=mouse_id, date=rec_date, probe=rec_probe,
            rf_session=rf_id, n_rf=panel['n_rf'], n_assessed=panel['n_assessed'],
            hd_session=hd_id, hd=file_record(hd_file),
            source=file_record(source), detection=file_record(result_path)))
        del maps
pooled_coverage = pool_count_coverage(coverage_panels)
assert len({item['mouse'] for item in coverage_sources}) == len(comparison_recordings) == 7
np.savez_compressed(output_dir / 'v9_seven_mice_population_data.npz', **pooled_coverage)
fig = count_coverage_figure(pooled_coverage)
save_figure(fig, 'v9_slide20_seven_mice_population_coverage')
display(fig)
plt.close(fig)

In [ ]:
# Slide 20: highlight the central −120° to +120° azimuth range.
fig = count_coverage_figure(pooled_coverage)
coverage_axis, azimuth_axis = fig.axes[:2]
for axis in (coverage_axis, azimuth_axis):
    # Place the tint above the opaque map and below the black marginal curve.
    axis.axvspan(-120, 120, color='#F4A261', alpha=.14,
                zorder=1.5, linewidth=0)
save_figure(fig, 'v9_slide20_seven_mice_population_coverage_highlight')
display(fig)
plt.close(fig)


## Original slide 20 (current slide 18) — seven-mouse all-HD fractions
Use the same seven mice as the coverage panel, including both M15 probes. The denominator is all saved HD class-3 units in the paired HD recordings. Count saved excitatory ON RF centers for each HD unit, matching the coverage panel and poster.ipynb. Keep RF-unassessed units separate from assessed zero-RF units; both remain in the all-HD denominator. The 129 RF-positive HD units are identical in the coverage map and fraction numerator. The denominator includes all 404 saved HD class-3 units.


In [ ]:
fraction_tables = []
fraction_sources = []
for mouse_id, rec_date, hd_id, rf_id, probes in comparison_recordings:
    for rec_probe in probes:
        rec_base = root / mouse_id / rec_date
        hd_file = rec_base / f'{rec_date}_{hd_id}/data/tuning_curves/Probe{rec_probe}/tuning_curves.tc'
        saved_hd = load_hd_tc(hd_file)
        ids = np.asarray(saved_hd['unit_id'])[np.asarray(saved_hd['unit_data']['hd_class']) == 3]
        table = pd.DataFrame(index=pd.Index(ids, name='unit_id'))
        rf_file = rf_source_path(rec_base / f'{rec_date}_{rf_id}', rec_probe)
        for kind in ('excitatory', 'inhibitory'):
            result_path = rf_result_path(rf_file, rf_type=kind)
            result = load_rf(result_path)
            table[kind] = pd.Series(result.center_2d.sum(axis=(1, 2)), index=result.unit_ids,
                                    dtype='Int64').reindex(table.index)
            fraction_sources.append(file_record(result_path))
        table['n_rf'] = table.excitatory
        table['category'] = 'RF unassessed'
        for selected, label in ((table.n_rf.eq(0), '0 RF'), (table.n_rf.eq(1), '1 RF'), (table.n_rf.gt(1), '>1 RF')):
            table.loc[selected.fillna(False), 'category'] = label
        table['mouse'], table['date'], table['probe'] = mouse_id, rec_date, rec_probe
        fraction_tables.append(table.reset_index())
        fraction_sources.append(file_record(hd_file))
fraction_units = pd.concat(fraction_tables, ignore_index=True)
fraction_counts = fraction_units.category.value_counts().reindex(['1 RF', '>1 RF', '0 RF', 'RF unassessed'], fill_value=0)
assert len(fraction_units) == fraction_counts.sum()
assert fraction_units.mouse.nunique() == len(comparison_recordings) == 7
assert not fraction_units.duplicated(['mouse', 'date', 'probe', 'unit_id']).any()
fraction_units.to_csv(output_dir / 'v9_slide20_seven_mice_hd_rf_fraction_units.csv', index=False)
fraction_counts.to_csv(output_dir / 'v9_slide20_seven_mice_hd_rf_fraction_counts.csv', header=['count'])
fig, axis = plt.subplots(figsize=(11, 1.6), layout='constrained', facecolor='white')
axis.barh(0, 100, color='white', edgecolor='black', height=.55)
left = 0
for category, color in (('1 RF', '#9CCAE5'), ('>1 RF', '#205A91')):
    percent = fraction_counts[category] / len(fraction_units) * 100
    axis.barh(0, percent, left=left, color=color, height=.55)
    axis.text(left + percent/2, 0, f'{category}: {percent:.1f}%', ha='center', va='center',
              color='white' if category == '>1 RF' else 'black')
    left += percent
axis.set(xlim=(0, 100), yticks=[], xticks=[], title=f'HD cells, n = {len(fraction_units)}')
axis.axis('off')
save_figure(fig, 'v9_slide20_seven_mice_hd_rf_fraction')
display(fig)
plt.close(fig)
# Keep a separate receipt so this slide can be regenerated without rerunning other figures.
fraction_by_recording = (fraction_units.groupby(['mouse', 'date', 'probe'], sort=False)
    .category.value_counts().unstack(fill_value=0).reindex(columns=fraction_counts.index, fill_value=0))
fraction_by_recording['n_hd'] = fraction_by_recording.sum(axis=1)
fraction_by_recording.to_csv(output_dir / 'v9_slide20_seven_mice_hd_rf_fraction_by_recording.csv')
n_hd_with_rf = int(fraction_counts['1 RF'] + fraction_counts['>1 RF'])
assert n_hd_with_rf == pooled_coverage['n_rf']
for panel, table in zip(coverage_panels, fraction_tables, strict=True):
    assert panel['n_rf'] == table.excitatory.gt(0).sum()
slide20_statistics = dict(
    notebook='slides_v2.ipynb', original_slide=20, current_slide=18,
    recordings=[dict(mouse=m, date=d, hd_session=h, rf_session=r, probes=list(p))
                for m, d, h, r, p in comparison_recordings],
    n_mice=len(comparison_recordings), coverage_n_rf=pooled_coverage['n_rf'],
    coverage_n_assessed=pooled_coverage['n_assessed'],
    coverage_definition='saved HD class 3 intersected with excitatory ON RF masks; no additional anatomy restriction',
    coverage_smoothing_sigma_deg=sigma_deg,
    marginal_definition='distinct RF-positive units per native azimuth column or elevation row, unsmoothed',
    fraction_n_hd=len(fraction_units), fraction_n_with_rf=n_hd_with_rf,
    fraction_with_rf=n_hd_with_rf / len(fraction_units),
    fraction_counts={key: int(value) for key, value in fraction_counts.items()},
    fraction_definition='all saved HD class-3 units; excitatory ON RF centers; unassessed retained separately',
    coverage_sources=coverage_sources, fraction_sources=fraction_sources)
(output_dir / 'v9_slide20_seven_mice_statistics.json').write_text(
    json.dumps(slide20_statistics, indent=2) + '\n')
display(fraction_by_recording)
print(f"7 mice: {pooled_coverage['n_rf']} RF-positive coverage units; "
      f"{n_hd_with_rf}/{len(fraction_units)} HD cells with RF ({n_hd_with_rf / len(fraction_units):.1%}).")


## Slides 22–24 — generate grayscale sorting panels
Use the same 18 cells in signed preferred-HD order. Normalize each response row by its maximum; zero is black and the maximum is white. RF values sum all elevation rows.

In [ ]:
hd_comparison = prep_hd_tc(load_hd_comparison(mouse, root, probe, date, hd_session), bins=30)
rf_comparison_all_height = prep_rf_tc(rf_profiles_saved, bins=30, max_zero_bins=None)
paired_keys = rf_comparison_all_height.index
hd_comparison = hd_comparison.loc[paired_keys]
signed_peaks = (peak_angles(hd_comparison, paired_keys) + 180) % 360 - 180
sorting_order = paired_keys[np.argsort(signed_peaks, kind='stable')].tolist()
sorting_profiles = {'HD': hd_comparison.loc[sorting_order], 'RF': rf_comparison_all_height.loc[sorting_order]}
sorting_rows = pd.DataFrame(dict(top_to_bottom_row=np.arange(1, 19),
    unit_id=[int(key.rsplit(':', 1)[1]) for key in sorting_order], unit_key=sorting_order))
sorting_rows.to_csv(output_dir / 'm22_261008_9_sorting_order.csv', index=False)
for kind, table in sorting_profiles.items():
    table.to_csv(output_dir / f'm22_261008_9_{kind.lower()}_profiles.csv')
    fig = sorting_figure(table, kind=kind)
    save_figure(fig, f'm22_261008_9_{kind.lower()}_visual_comparison_panel')
    display(fig)
    plt.close(fig)

## Slide 23 — generate the 2D-to-1D example
Unit 269 must retain a peak of 545 pooled counts at −66°. All seven rows contribute. The normalized curve and heatmap row contain identical values.

In [ ]:
projection_example = projection_data(example_counts[269]['ON'], on, example_responses[269]['ON'])
np.testing.assert_array_equal(projection_example['selected_rows'], np.arange(7))
np.testing.assert_array_equal(projection_example['native_counts'], projection_example['counts_2d'].sum(axis=0))
np.testing.assert_array_equal(projection_example['native_counts'], rf_profiles_saved.loc['m22:261008:A:269'])
np.testing.assert_allclose(projection_example['normalized'], sorting_profiles['RF'].loc['m22:261008:A:269'])
assert projection_example['native_counts'].max() == 545
assert projection_example['x'][projection_example['native_counts'].argmax()] == -66
stem = 'm22_261008_unit269_rf_2d_to_1d'
np.savez_compressed(output_dir / f'{stem}.npz', **projection_example)
(output_dir / f'{stem}.json').write_text(json.dumps(
    projection_export(projection_example, {'ON': file_record(rf_paths['ON']),
        'saved_detection': file_record(detector_paths['ON'])}, cmap='gray'), indent=2) + '\n')
fig = projection_figure(projection_example, cmap='gray', line_color='black')
save_figure(fig, stem)
display(fig)
plt.close(fig)

## Sample unit — RF polar map with a forward-facing mouse
M22 unit 269, ON response in 0–200 ms. Reuse the native annular RF map, firing-rate scale and saved RF outline. The mouse faces 0° (up); azimuth increases clockwise, and the rings encode elevation.


In [ ]:
sample_rf_polar_stem = 'm22_261008_unit269_rf_polar_mouse'
fig = sample_rf_polar_figure(example_responses[269]['ON'], example_masks[269]['ON'], 269)
save_figure(fig, sample_rf_polar_stem)
display(fig)
plt.close(fig)
sample_rf_polar_files = [output_dir / f'{sample_rf_polar_stem}.{extension}'
                         for extension in ('svg', 'png')]


## Slides 25–26 — generate schematics and region-center plots
All six figures are drawn directly with Matplotlib; no SVG templates or image assets are required. The same 25 saved excitatory RF regions from 18 cells feed both arrow variants and the histogram. Red arrows in the highlighted variant mark directions within 30° of 48°. Several regions can come from one cell. The numerical angle sum is not a calibrated world-coordinate transform.


In [ ]:
angles_rad = np.deg2rad(region_pairs.direction_deg.to_numpy())
region_counts, region_edges = np.histogram(angles_rad, bins=histogram_bins, range=(0, 2*np.pi))
region_bin_centers_deg = np.rad2deg((region_edges[:-1] + region_edges[1:]) / 2)
region_score, region_p = rayleigh_test(region_counts, np.ones_like(region_counts), region_bin_centers_deg)
assert region_counts.sum() == len(region_pairs) == 25
region_statistics = dict(
    mouse=mouse, date=date, probe=probe, hd_session=hd_session, rf_session=rf_session,
    n_cells=n_cells, n_regions=n_regions, rf_type='excitatory',
    hd_bins=arrow_hd_bins, hd_smoothing_sigma_deg=arrow_hd_smoothing_deg,
    definition='native HD peak + each saved excitatory RF center X azimuth, modulo 360',
    counts=region_counts.tolist(), edges_deg=np.rad2deg(region_edges).tolist(),
    rayleigh_score=region_score, rayleigh_p_nominal=region_p,
    method='Existing slide method: 15-bin histogram, equal occupancy, Rayleigh score test',
    interpretation='Descriptive region-level p; regions from the same cell are not independent',
    regions_per_cell={str(unit): int(count) for unit, count in region_pairs.groupby('unit_id').size().items()},
)
(output_dir / 'm22_261008_hd_rf_25_region_statistics.json').write_text(json.dumps(region_statistics, indent=2) + '\n')
print(f'{n_regions} RF regions / {n_cells} cells; Rayleigh score = {region_score:.6f}; nominal p = {region_p:.8f}')
from scripts.slides_v9_figures import (
    schematic_figure, region_positions_figure, region_histogram_figure,
)

schematic_figures = ('schematic_hd', 'schematic_rf', 'schematic_combined',
                     'hd_positions_original_25_regions', 'hd_positions_25_regions',
                     'hd_rf_25_region_histogram')
for kind in ('hd', 'rf', 'combined'):
    fig = schematic_figure(kind)
    save_figure(fig, f'm22_261008_schematic_{kind}')
    plt.close(fig)
for stem, highlighted in (('hd_positions_original_25_regions', False),
                           ('hd_positions_25_regions', True)):
    fig = region_positions_figure(region_pairs, highlighted=highlighted)
    save_figure(fig, f'm22_261008_{stem}')
    plt.close(fig)
fig = region_histogram_figure(region_statistics)
save_figure(fig, 'm22_261008_hd_rf_25_region_histogram')
plt.close(fig)
region_figure_files = [output_dir / f'm22_261008_{stem}.{extension}'
                      for stem in schematic_figures for extension in ('svg', 'png')]
print(f'Generated {len(schematic_figures)} figures as SVG and PNG directly from code.')


In [ ]:
for stem in schematic_figures:
    display(Image(filename=str(output_dir / f'm22_261008_{stem}.png')))


## Slide 26 — add a peak-bin polar plot
One observation per cell, as in the TC comparison notebook. Use each 30-bin HD curve maximum and its all-height RF curve maximum. This is separate from the 25-region-center histogram.

In [ ]:
m22_peak_data = peak_sum_data(sorting_profiles['HD'], sorting_profiles['RF'], bins=histogram_bins)
assert m22_peak_data['counts'].sum() == 18
m22_peak_data['peaks'].to_csv(output_dir / 'm22_261008_hd_rf_peak_bin_pairs.csv', index_label='unit_id')
fig = peak_sum_figure(m22_peak_data)
save_figure(fig, 'v9_slide26_hd_rf_peak_bins_polar')
display(fig)
plt.close(fig)

## New slide 27 — peak-bin polar plots for seven mice
M14, M15, M19, M21, M22, M23, M24, in that order. Select saved HD class 3 intersected with units that have at least one saved excitatory ON RF center. Apply the same selection to all mice without anatomical restrictions. RF peak values then use all-height raw counts. M15 pools both probes using full recording/unit keys. No M20 or repeated M19 date is included.

In [ ]:
seven_mice_data, seven_mice_sources = {}, []
for mouse_id, rec_date, hd_id, rf_id, probes in comparison_recordings:
    hd_tables, rf_tables = [], []
    for rec_probe in probes:
        rec_base = root / mouse_id / rec_date
        hd_table = prep_hd_tc(load_hd_comparison(mouse_id, root, rec_probe, rec_date, hd_id), bins=30)
        rf_range = [-150, 150] if mouse_id == 'm14' else [-180, 180]
        source = rf_source_path(rec_base / f'{rec_date}_{rf_id}', rec_probe)
        detection = load_rf(rf_result_path(source, rf_type='excitatory'))
        rf_ids = detection.unit_ids[detection.center_2d.any(axis=(1, 2))]
        rf_keys = [f'{mouse_id}:{rec_date}:{rec_probe}:{unit}' for unit in rf_ids]
        keys = hd_table.index.intersection(rf_keys)
        maps = load_rfmap(source)
        raw = pd.DataFrame(
            [maps.by_unit_id(int(key.rsplit(':', 1)[1])).sum(0, .2)
             .sum_to_1d(axis='x', rf_only=False).to_1d_array() for key in keys],
            index=keys, columns=maps[0].x_positions)
        raw.attrs.update(label=f'{mouse_id.upper()} all-height RF', range=rf_range,
                         response_units='spike_count', response_kind='RF')
        raw.to_csv(output_dir / f'{mouse_id}_{rec_date}_Probe{rec_probe}_all_height_native_counts.csv', index_label='unit_id')
        hd_tables.append(hd_table.loc[keys])
        rf_tables.append(prep_rf_tc(raw, bins=30, max_zero_bins=None))
        seven_mice_sources.append(dict(mouse=mouse_id, date=rec_date, probe=rec_probe,
            hd_session=hd_id, rf_session=rf_id, n_cells=len(keys), source=file_record(source)))
        del maps
    hd_mouse = concat(*hd_tables, label=f'{mouse_id.upper()} HD')
    rf_mouse = concat(*rf_tables, label=f'{mouse_id.upper()} all-height RF')
    data = peak_sum_data(hd_mouse, rf_mouse, bins=histogram_bins)
    seven_mice_data[mouse_id] = data
    data['peaks'].to_csv(output_dir / f'{mouse_id}_hd_rf_peak_bin_pairs.csv', index_label='unit_id')
    if mouse_id == 'm22':
        np.testing.assert_array_equal(data['counts'], m22_peak_data['counts'])
np.savez_compressed(output_dir / 'v9_seven_mice_peak_histograms.npz',
    edges=next(iter(seven_mice_data.values()))['edges'],
    **{f'{mouse_id}_counts': data['counts'] for mouse_id, data in seven_mice_data.items()})
fig = seven_mice_figure(seven_mice_data)
save_figure(fig, 'v9_slide27_seven_mice_peak_bins_polar')
display(fig)
plt.close(fig)
display(pd.DataFrame(seven_mice_sources))

## Source record and final checks
Raw data hashes, cohort identities, region counts, histogram bins and exact RF aggregation are saved alongside the exports.

In [ ]:
assert list(seven_mice_data) == ['m14', 'm15', 'm19', 'm21', 'm22', 'm23', 'm24']
assert region_counts.sum() == n_regions == 25
assert all(path.is_file() for path in region_figure_files + sample_rf_polar_files)
for unit, count in region_pairs.groupby('unit_id').size().items():
    assert count == on.for_unit(int(unit), center_only=True).sum()
manifest = dict(notebook='slides_v2.ipynb', source_deck='WorkInProgress_v9.pptx',
    m22_sources={key: file_record(path) for key, path in {'HD': hd_path, **rf_paths,
        **{f'{key}_detection': value for key, value in detector_paths.items()}}.items()},
    paired_m22_unit_keys=sorting_order, sample_unit_ids=example_units,
    coverage_sources=coverage_sources, fraction_sources=fraction_sources,
    seven_mice_sources=seven_mice_sources,
    seven_mice_statistics={mouse_id: dict(n_cells=int(data['counts'].sum()),
        counts=data['counts'].tolist(), rayleigh_score=float(data['rayleigh_score']),
        rayleigh_p=float(data['rayleigh_p'])) for mouse_id, data in seven_mice_data.items()},
    rf_aggregation='raw 0-200 ms counts summed over every elevation row',
    cohort_selection='saved HD class 3 intersected with saved excitatory ON RF centers; no anatomical restriction',
    peak_definition='30-bin curve maximum, first source bin on ties; one pair per cell',
    histogram_bins=histogram_bins, region_count=n_regions,
    region_figure_rendering='Matplotlib; code-drawn schematics and measured region table',
    region_figure_files=[str(path) for path in region_figure_files],
    sample_rf_polar=dict(unit_id=269, polarity='ON', mouse_heading_deg=0,
        files=[str(path) for path in sample_rf_polar_files]),
    polar_color='#266c9b',
    native_count_unit269_peak=545, grayscale='black=zero, white=maximum',
    fraction_n_hd=len(fraction_units), fraction_counts=fraction_counts.to_dict(),
    pooled_coverage_n_rf=pooled_coverage['n_rf'])
(output_dir / 'v9_figure_sources.json').write_text(json.dumps(manifest, indent=2) + '\n')
print(f'Generated v9 figure data and SVG/PNG outputs in {output_dir}')

## Slides 25–26 — the same RF-region direction analysis across seven mice
This section extends the **25-region M22 arrow plot and histogram** to M14, M15, M19, M21, M22, M23 and M24. Each observation is one saved excitatory RF center, including multiple centers from a cell. Its direction is the native **180-bin HD maximum after circular Gaussian smoothing (σ = 12°) of counts and occupancy**, plus the RF center X azimuth, modulo 360°. All histograms use the same 15 bins (24° each). This is separate from the preceding one-peak-per-cell analysis.

Select saved HD class 3 intersected with units that have at least one saved excitatory ON RF center, without anatomical restrictions. Pool M15 probes A and B using full recording/unit keys. Loading, selection, statistics and explicit export are separate cells. Region-level Rayleigh p values are descriptive because regions from the same cell are not independent. The numerical angle sum is not a calibrated world-coordinate transform.


In [ ]:
# Read saved inputs only. Do not regenerate tuning curves or RF detections.
region_inputs = []
for mouse_id, rec_date, hd_id, rf_id, probes in comparison_recordings:
    for rec_probe in probes:
        rec_base = root / mouse_id / rec_date
        region_hd_path = rec_base / f'{rec_date}_{hd_id}/data/tuning_curves/Probe{rec_probe}/tuning_curves.tc'
        region_rf_path = rf_source_path(rec_base / f'{rec_date}_{rf_id}', rec_probe)
        region_detection_path = rf_result_path(region_rf_path, rf_type='excitatory')
        region_hd_saved = load_hd_tc(region_hd_path)
        region_detected = load_rf(region_detection_path)
        region_source_files = dict(HD=file_record(region_hd_path),
                                   RF_detection=file_record(region_detection_path))
        assert region_detected.manifest['parameters']['alternative'] == 'greater'
        np.testing.assert_allclose(region_detected.manifest['time_range_s'], [0, .2])
        region_inputs.append(dict(mouse=mouse_id, date=rec_date, probe=rec_probe,
            hd_session=hd_id, rf_session=rf_id, hd=region_hd_saved, rf=region_detected,
            source_files=region_source_files))
print(f'Loaded {len(region_inputs)} recording/probe pairs for seven mice.')


In [ ]:
# Select HD class 3 with saved ON RF centers, then preserve native coordinates.
region_mouse_tables, region_probe_audit = {}, []
for entry in region_inputs:
    mouse_id, rec_date, rec_probe = entry['mouse'], entry['date'], entry['probe']
    region_hd_30 = hd_profiles(entry['hd'], probe=rec_probe, bins=30, smoothing_deg=0, hd_class=3)
    region_hd_30.index = region_hd_30.index.get_level_values('unit_id')
    region_rf_ids = pd.Index(entry['rf'].unit_ids[entry['rf'].center_2d.any(axis=(1, 2))])
    region_selected_ids = region_hd_30.index.intersection(region_rf_ids)
    region_selected_ids = region_hd_30.loc[region_selected_ids].idxmax(axis=1).sort_values(kind='stable').index
    region_hd_native = hd_profiles(entry['hd'], probe=rec_probe, bins=arrow_hd_bins,
                                   smoothing_deg=arrow_hd_smoothing_deg, hd_class=3)
    region_hd_native.index = region_hd_native.index.get_level_values('unit_id')
    region_hd_peaks = region_hd_native.loc[region_selected_ids].idxmax(axis=1)
    region_detection = entry['rf']
    region_centers = rf_centers(region_detection,
        x_positions=region_detection.manifest['x_positions'],
        y_positions=region_detection.manifest['y_positions'], probe=rec_probe)
    region_table = region_centers.loc[pd.MultiIndex.from_product([[rec_probe], region_selected_ids])].reset_index()
    region_table['rf_region'] = region_table.groupby('unit_id', sort=False).cumcount() + 1
    region_table['hd_peak_deg'] = region_table.unit_id.map(region_hd_peaks)
    region_table['direction_deg'] = (region_table.hd_peak_deg + region_table.rf_x_deg) % 360
    region_table['rf_type'] = 'excitatory'
    region_table['mouse'], region_table['date'] = mouse_id, rec_date
    region_table['hd_session'], region_table['rf_session'] = entry['hd_session'], entry['rf_session']
    region_table['unit_key'] = [f'{mouse_id}:{rec_date}:{rec_probe}:{unit}' for unit in region_table.unit_id]
    assert region_table.unit_id.nunique() == len(region_selected_ids)
    for unit, count in region_table.groupby('unit_id').size().items():
        assert count == region_detection.for_unit(int(unit), center_only=True).sum()
    region_mouse_tables.setdefault(mouse_id, []).append(region_table)
    region_probe_audit.append(dict(mouse=mouse_id, date=rec_date, probe=rec_probe,
        hd_session=entry['hd_session'], rf_session=entry['rf_session'],
        n_cells=len(region_selected_ids), n_regions=len(region_table), sources=entry['source_files']))
region_mouse_tables = {mouse_id: pd.concat(tables, ignore_index=True)
                       for mouse_id, tables in region_mouse_tables.items()}
region_m22_check = region_mouse_tables['m22'].loc[:, region_pairs.columns]
pd.testing.assert_frame_equal(region_m22_check.reset_index(drop=True), region_pairs.reset_index(drop=True))
print('M22 matches every slide-25 region, unit, HD peak and direction exactly.')


In [ ]:
# Calculate every mouse with the exact histogram definition used for M22 arrows.
region_mouse_statistics = {}
for mouse_id, region_table in region_mouse_tables.items():
    region_values_rad = np.deg2rad(region_table.direction_deg.to_numpy())
    region_hist_counts, region_hist_edges = np.histogram(region_values_rad, bins=histogram_bins, range=(0, 2*np.pi))
    region_hist_centers = np.rad2deg((region_hist_edges[:-1] + region_hist_edges[1:]) / 2)
    region_hist_score, region_hist_p = rayleigh_test(region_hist_counts, np.ones_like(region_hist_counts), region_hist_centers)
    assert region_hist_counts.sum() == len(region_table)
    region_mouse_statistics[mouse_id] = dict(mouse=mouse_id, date=str(region_table.date.iloc[0]),
        hd_session=int(region_table.hd_session.iloc[0]), rf_session=int(region_table.rf_session.iloc[0]),
        probes=region_table.probe.unique().tolist(), n_cells=region_table.unit_key.nunique(),
        n_regions=len(region_table), counts=region_hist_counts.tolist(),
        edges_deg=np.rad2deg(region_hist_edges).tolist(),
        rayleigh_score=float(region_hist_score), rayleigh_p_nominal=float(region_hist_p),
        regions_per_cell={key: int(count) for key, count in region_table.groupby('unit_key').size().items()})
assert list(region_mouse_statistics) == ['m14', 'm15', 'm19', 'm21', 'm22', 'm23', 'm24']
assert (region_mouse_statistics['m22']['n_cells'], region_mouse_statistics['m22']['n_regions']) == (18, 25)
np.testing.assert_array_equal(region_mouse_statistics['m22']['counts'], [1, 9, 2, 2, 0, 0, 0, 0, 3, 3, 2, 0, 0, 0, 3])
np.testing.assert_array_equal(region_mouse_statistics['m22']['counts'], region_counts)
region_comparison_manifest = dict(
    definition='native HD peak + each saved excitatory RF center X azimuth, modulo 360',
    hd_bins=arrow_hd_bins, hd_smoothing_sigma_deg=arrow_hd_smoothing_deg, bins=histogram_bins,
    hd_preparation='Circular Gaussian smoothing of native spike counts and occupancy separately, then rate division; first source bin on ties',
    cohort_selection='Saved HD class 3 intersected with saved excitatory ON RF centers; no anatomical restriction',
    interpretation='One observation per saved RF region; region-level Rayleigh p is descriptive because regions from the same cell are not independent',
    mice=region_mouse_statistics, recording_sources=region_probe_audit)
display(pd.DataFrame([{key: value for key, value in stats.items()
                      if key in ('mouse', 'n_cells', 'n_regions', 'rayleigh_score', 'rayleigh_p_nominal')}
                     for stats in region_mouse_statistics.values()]))


In [ ]:
# Explicitly save the region data separately from the preceding peak-bin outputs.
from scripts.slides_v9_figures import seven_mice_region_figure
for mouse_id, region_table in region_mouse_tables.items():
    region_table.to_csv(output_dir / f'{mouse_id}_hd_rf_region_center_pairs.csv', index=False)
    (output_dir / f'{mouse_id}_hd_rf_region_center_statistics.json').write_text(
        json.dumps(region_mouse_statistics[mouse_id], indent=2) + '\n')
pd.concat(region_mouse_tables.values(), ignore_index=True).to_csv(
    output_dir / 'v9_seven_mice_region_direction_pairs.csv', index=False)
(output_dir / 'v9_seven_mice_region_direction_statistics.json').write_text(
    json.dumps(region_comparison_manifest, indent=2) + '\n')
fig = seven_mice_region_figure(region_mouse_statistics)
save_figure(fig, 'v9_slide26_seven_mice_region_centers_polar')
display(fig)
plt.close(fig)
print('Saved seven-mouse RF-region directions, source provenance and SVG/PNG plots.')
